# Week 2 Day 5 - Food Service

We'll now bring together what we've learned to make an AI Customer Support assistant for a Food Service
1. A multi-modal AI assistant with image and audio generation
2. Tool callling with database menu lookup (you may use day5_db_foodservice.ipynb to create foodservice_menu)
3. A step towards an Agentic workflow

In [ ]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "foodservice_menu.db"

In [ ]:
SYSTEM_PROMPT = """You are a helpful food services assistant for a Food Truck or Restaurant (test data only, no real orders or charges). 
You will ask if they prefer to order from food service, Food Truck or Restaurant, where the menu details are in a DB.
They can order from Food Truck or Restaurant and not both. 

Guidelines:
- Before completing an order, confirm the chosen option and total price with the user 
- Always call get_menu before a conversation if you haven't already seen the menu details in this conversation,
 ask the customer if they prefer a food truck or restaurant.
- After call_menu, present the menu items clearly with option numbers, category, item name, description and price. Customer can select more than 1.
- After get_menu, present the menu clearly with menu category, item_name, description, and price.
- Never invent menu data — only report what the tools return.
- Remind the user this is sandbox/test data, not a real order, the first time you book something.
"""

In [ ]:

def get_menu(food_service: str) -> str:
    print(f"DATABASE TOOL CALLED: Getting menu for {food_service}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('''
            SELECT food_service, category, item_name, description, price 
            FROM menu_items 
            WHERE food_service = ?
        ''', (food_service,))
        
        # Use fetchall to extract ALL matching rows
        results = cursor.fetchall()
        
        if not results:
            return f"No menu available for '{food_service}'"
            
        # Format the list of items into a structured text block
        menu_lines = [f"Menu for {food_service}:"]
        for row in results:
            # Unpacking the tuple elements from the query
            _, category, item_name, description, price = row
            menu_lines.append(f" - [{category}] {item_name}: {description} (${price:.2f})")
            
        return "\n".join(menu_lines)


In [ ]:
food_service_function = {
    "name": "get_menu",
    "description": "Get the menu for a specific food service type.",
    "parameters": {
        "type": "object",
        "properties": {
            "food_service": {
                "type": "string",
                "description": "The type of food service provider requested.",
                "enum": ["food truck", "restaurant"]  # Forces the AI to pick exactly one of these
            },
        },
        "required": ["food_service"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": food_service_function}]


In [ ]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image

In [ ]:
def artist(food_service):
    image_response = openai.images.generate(
            model="gpt-image-1-mini",
            prompt=f"An image representing a {food_service} of the future, named FOODY AI, showing a futuristic tourist spot"
            size="1024x1024",
            n=1,
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))


In [ ]:
image = artist("food truck")
display(image)

In [ ]:
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

In [ ]:
def chat(history, current_service):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    services = []

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, turn_services = handle_tool_calls_and_return_services(message)
        services.extend(turn_services)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    # 1) Show the reply text right away. Audio/image aren't ready yet, so skip
    #    them for now -> the chat no longer waits on TTS before rendering.
    yield history, gr.skip(), gr.skip(), current_service

    # 2) Now do the slow work: speech every turn, image only on a new service.
    voice = talker(reply)
    image_update = gr.skip()
    if services and services[-1] != current_service:
        current_service = services[-1]
        image_update = artist(current_service)

    yield history, voice, image_update, current_service


In [ ]:
def handle_tool_calls_and_return_services(message):
    responses = []
    services = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_menu":
            arguments = json.loads(tool_call.function.arguments)
            food_service = arguments.get('food_service')
            services.append(food_service)
            menu_details = get_menu(food_service)
            responses.append({
                "role": "tool",
                "content": menu_details,
                "tool_call_id": tool_call.id
            })
    return responses, services

In [ ]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
    return "", history + [{"role":"user", "content":message}]

# Generate the default image only ONCE per process and cache it at module level.
# Every session's ui.load reuses this same picture, so the inline (Cursor) and
# inbrowser views stay in sync instead of each drawing a fresh, different image.
_default_image = None

def load_default_image():
    global _default_image
    if _default_image is None:
        _default_image = artist(
            "An image representing food truck and restaurant of the future, showing a futuristic tourist spot"
        )
    # start with no selected service so the first real choice always regenerates
    return _default_image, None

# UI definition

with gr.Blocks() as ui:
    service_state = gr.State(None)          # remembers the currently shown service

    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False,
                                show_download_button=True, label="FOODY AI")
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    # Cached default image, shown on load
    ui.load(load_default_image, inputs=None, outputs=[image_output, service_state])

    # Chat regenerates the image only when a new service is chosen
    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=[chatbot, service_state],
        outputs=[chatbot, audio_output, image_output, service_state]
    )

# ui.launch(inbrowser=True, auth=("ed", "bananas"))
ui.launch(inbrowser=True)
